<table style="width:100%; border:2px solid #2583d8; border-radius:10px; background-color:white;">
<tr>

<td style="width:34%; vertical-align:middle; padding:20px;">
<img src="./assets/RoboDedito.png"
     alt="Robot character"
     style="width:100%; max-width:480px; display:block; margin:auto; border-radius:10px;">
</td>

<td style="width:66%; vertical-align:middle; padding:25px 35px;">

<h2 style="color:#1474c4; font-size:30px;">
Comparing LLM Models.
</h2>

<p style="font-size:20px;"> Create a new folder for this project.</p>

<p style="font-size:20px;"> Open the folder using <code>VSCode<code>.</p>

<p style="font-size:20px;"> The uv package was already installed on the local computer (see first lab)</p>

<p style="font-size:20px;"> Create a virtual environment. Run the command <code>uv init</code> to start <code>uv<code></p>

<p style="font-size:20px;"> For this lab, we will need an OpenAI API key.</p>

<p style="font-size:20px;"> Create another file named <code>.env</code> and place that key inside.</p>

<p style="font-size:20px;"> <em>OPENAI_API_KEY=sk-proj-l6QOo</em> </p>

</td>
</tr>
</table>

<div style="margin-top:20px; border:2px solid #2583d8; border-radius:10px; background-color:#ffffff; padding:25px 30px; font-family:Arial, sans-serif;">

<p style="font-size:28px;"> Description.</p>
<p style="font-size:20px;"> This project is about asking several LLMs the same question and collecting their answers.</p>
<p style="font-size:20px;"> Then use another LLM as a judge to rank the responses. </p>
<p style="font-size:20px;"> The first LLM will be OpenAI, and the rest will be several open-source models that can run in <code>Ollama.</code></p>
<p style="font-size:20px;"> I will also use OpenAI to judge the answers.</p>
<p style="font-size:20px;"> Ollama runs a local web service that gives an OpenAI-compatible endpoint and runs models locally using high-performance C++ code.</p>
<p style="font-size:20px;"> If you don't have Ollama, install it here by visiting https://ollama.com.</p>
<p style="font-size:20px;"> After it's installed, you should be able to visit here: http://localhost:11434 and see the message "Ollama is running"</p>
<p style="font-size:20px;"> I will test the models: <em>deepseek-r1:14b and gemma4:latest</em> </p>

</div>

<div style="margin-top:20px; border:2px solid #2583d8; border-radius:10px; background-color:#ffffff; padding:25px 30px; font-family:Arial, sans-serif;">

<p style="font-size:28px;"> Script in Python</p>
<p style="font-size:20px;"> The next script is how this project is deployed</p>
<p style="font-size:20px;"> I have added comments for better understanding</p>

</div>

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

# These are the model names used in the notebook's code cells.
QUESTION_AND_JUDGE_MODEL = "gpt-5-mini"
OPENAI_COMPETITOR = "gpt-4o-mini"
OLLAMA_MODELS = [
    "deepseek-r1:14b",
    "gemma4:latest",]

def ask_model(client, model_name, messages):
    """Send a conversation to a model and return its text response."""
    response = client.chat.completions.create(
        model=model_name,
        messages=messages,)
    # The API wraps the generated text inside the first response choice.
    answer = response.choices[0].message.content
    if not answer:
        raise RuntimeError(f"Model {model_name} returned no text.")
    return answer

def main():
    # Resolve .env relative to this file, even when launched from another folder.
    # .env values override existing environment variables.
    load_dotenv(Path(__file__).resolve().with_name(".env"), override=True)
    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError(
            "OPENAI_API_KEY is not set. Add it to your environment or .env file.")
    print("OpenAI API key is configured.")

    # Both services accept the same client interface. Ollama uses a placeholder
    # key because the client requires one for its local endpoint.
    openai = OpenAI()
    ollama = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

    # 1. Generate one question to use for every competitor.
    request = (
        "Please come up with a simple question that I can ask "
        "a number of LLMs to evaluate their intelligence. "
        "Answer only with the question, no explanation.")
    print("\nGenerating the evaluation question...", flush=True)
    question = ask_model(
        openai, QUESTION_AND_JUDGE_MODEL, [{"role": "user", "content": request}])
    print(f"\nQuestion:\n{question}")

    # 2. Ask each model the same question, without showing it the other answers.
    # Matching positions in these lists associate each model with its answer.
    competitors = []
    answers = []
    messages = [{"role": "user", "content": question}]
    model_clients = [(OPENAI_COMPETITOR, openai)]
    model_clients.extend((model_name, ollama) for model_name in OLLAMA_MODELS)

    for model_name, client in model_clients:
        print(f"\nAsking {model_name}...", flush=True)
        answer = ask_model(client, model_name, messages)
        print(answer)
        competitors.append(model_name)
        answers.append(answer)
    together = "\n\n".join(
        f"# Response from competitor {competitor}\n\n{answer}"
        for competitor, answer in zip(competitors, answers))    

    judge = f"""You are judging a competition between {len(competitors)} competitors.
Each model has been given this question:

{question}

Your job is to evaluate each response for clarity and strength of argument, and rank them in order of best to worst.
Here are the responses from each competitor:

{together}

Now respond with a short description about why you gave each ranking."""

    # 4. Ask the judge to return competitor numbers from best to worst.
    print("\nJudging the answers...", flush=True)
    results = ask_model(
        openai, QUESTION_AND_JUDGE_MODEL, [{"role": "user", "content": judge}])
    
    print("\nJudge's full response:")
    print(results)

# Run the workflow only when launched as a script, not when imported elsewhere.
if __name__ == "__main__":
    main()

<div style="margin-top:20px; border:2px solid #2583d8; border-radius:10px; background-color:#ffffff; padding:25px 30px; font-family:Arial, sans-serif;">

<p style="font-size:28px;"> Improving the script</p>
<p style="font-size:20px;"> We can use LiteLLm to orchestrate the diferent types of LLMs models.</p>

</div>

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
from litellm import completion

# -------------------
# 1. PROJECT SETUP
# -------------------

# Gets the folder where this Python script is located.
BASE_DIR = Path(__file__).resolve().parent

# Loads environment variables from the .env file.
load_dotenv(BASE_DIR / ".env", override=True)

# ---------------------------------------------------------
# 2. API KEYS SETUP
# ---------------------------------------------------------

# Reads the OpenAI API key from the environment.
openai_api_key = os.getenv("OPENAI_API_KEY")

# Stops the program if the OpenAI API key is missing.
if not openai_api_key:
    raise RuntimeError("OPENAI_API_KEY is not set in the .env file.")

# LiteLLM reads provider API keys from environment variables.
os.environ["OPENAI_API_KEY"] = openai_api_key.strip()

# ---------------------------------------------------------
# 3. MODEL NAMES
# --------------------------------------------------------- 

QUESTION_AND_JUDGE_MODEL = "openai/gpt-5-mini"
OPENAI_COMPETITOR = "openai/gpt-4o-mini"
OLLAMA_MODELS = [
    "ollama/deepseek-r1:14b",
    "ollama/gemma4:latest",]

def ask_model(model_name, messages):
    """Send a conversation to a model and return its text response."""

    response = completion(
        model=model_name,
        messages=messages,)

    answer = response.choices[0].message.content
    if not answer:
        raise RuntimeError(f"Model {model_name} returned no text.")
    return answer

def main():
    
    # 1. Generate one question to use for every competitor.
    request = (
        "Please come up with a simple question that I can ask "
        "a number of LLMs to evaluate their intelligence. "
        "Answer only with the question, no explanation.")
    print("\nGenerating the evaluation question...", flush=True)
    question = ask_model(QUESTION_AND_JUDGE_MODEL, [{"role": "user", "content": request}])
    print(f"\nQuestion:\n{question}")

    # 2. Ask each model the same question, without showing it the other answers.
    # Matching positions in these lists associate each model with its answer.
    competitors = []
    answers = []
    messages = [{"role": "user", "content": question}]
    model_clients = [(OPENAI_COMPETITOR)]
    model_clients.extend((model_name) for model_name in OLLAMA_MODELS)

    for model_name in model_clients:
        print(f"\nAsking {model_name}...", flush=True)
        answer = ask_model(model_name, messages)
        print(answer)
        competitors.append(model_name)
        answers.append(answer)
    together = "\n\n".join(
        f"# Response from competitor {competitor}\n\n{answer}"
        for competitor, answer in zip(competitors, answers))    

    judge = f"""You are judging a competition between {len(competitors)} competitors.
Each model has been given this question:

{question}

Your job is to evaluate each response for clarity and strength of argument, and rank them in order of best to worst.
Here are the responses from each competitor:

{together}

Now respond with a short description about why you gave each ranking."""

    # 4. Ask the judge to return competitor numbers from best to worst.
    print("\nJudging the answers...", flush=True)
    results = ask_model(
        QUESTION_AND_JUDGE_MODEL, [{"role": "user", "content": judge}])
    
    print("\nJudge's full response:")
    print(results)

# Run the workflow only when launched as a script, not when imported elsewhere.
if __name__ == "__main__":
    main()

<table style="width:100%; border:2px solid #2583d8; border-radius:10px; background-color:white;">
<tr>

<td style="width:34%; vertical-align:middle; padding:20px;">
<img src="assets/RoboShowi.png"
     alt="Robot showing"
     style="width:100%; max-width:480px; display:block; margin:auto; border-radius:10px;">
</td>

<td style="width:66%; vertical-align:middle; padding:25px 35px;">

<h2 style="color:#1474c4; font-size:30px;">
That's it !!!
</h2>

<p style="font-size:20px;"> Now you have:</p>

<p style="font-size:20px;"> Your own virtual assistant.</p>

<p style="font-size:20px;"> You can use paid llms or.</p>

<p style="font-size:20px;"> Open-souce models. </p>

<p style="font-size:20px;"> Everything in a Python LLM application using SDK. </p>

</td>
</tr>
</table>